# The IAEA-2D PWR benchmark

The canonical coarse-mesh verification problem for a nodal diffusion code: a
quarter core of 20 cm assemblies, two energy groups, five compositions, with a
published reference eigenvalue of **1.02959**.

It is the problem that shows what a nodal method buys you. On one node per
assembly — a 20 cm mesh — finite difference is 373 pcm high, while the
semi-analytic nodal kernel is within 4 pcm of the reference.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import openndm

sys.path.insert(0, str(Path.cwd().parent / "benchmarks"))
from common import IAEA_RADIAL_MAP, check_radial_map, iaea_library

REFERENCE = 1.02959
print("openndm", openndm.__version__)

## The core map

Row 0 and column 0 lie on the symmetry lines, so the map is a quarter core.
The numbers are 1-based composition indices; 0 marks a position outside the
core.

In [ ]:
print(IAEA_RADIAL_MAP)
check_radial_map()
print("\nmap invariants hold: symmetric about the diagonal, and the")
print("peripheral fuel-1 band is edge-connected")

Those two invariants are asserted rather than assumed because the first
transcription of this deck violated both, and neither violation had any
symptom other than the eigenvalue. A single dropped cell in the peripheral
band was worth 80 pcm.

| Composition | Meaning | Σa2 |
|---|---|---|
| 1 | fuel 1, outer ring | 0.080 |
| 2 | fuel 2, interior | 0.085 |
| 3 | fuel 2 with a control rod | 0.130 |
| 4 | reflector | 0.010 |
| 5 | reflector with a rod | 0.055 |

## Build and solve

In [ ]:
def build(subdivide=1):
    core = np.where(IAEA_RADIAL_MAP == 0,
                    openndm.INACTIVE, IAEA_RADIAL_MAP - 1)[np.newaxis, :, :]
    return openndm.Geometry.from_lattice(
        core, pitch=(20.0, 20.0, 20.0), subdivide=(subdivide, subdivide, 1),
        boundaries={"x_min": "reflective", "y_min": "reflective",
                    "x_max": "zero_flux", "y_max": "zero_flux",
                    "z_min": "reflective", "z_max": "reflective"},
        outside="zero_flux")

library = iaea_library()
settings = openndm.Settings(
    verbosity=0, k_tolerance=1e-11, fission_source_tolerance=1e-10,
    inner_tolerance=1e-9, max_inner=400, max_outer=5000)

geom = build(1)
result = openndm.Model(geom, library, settings).solve(kernel="sanm")
print(f"SANM, one node per assembly ({geom.n_nodes} nodes)")
print(f"  k_eff = {result.k_eff:.6f}   "
      f"{1e5 * (result.k_eff - REFERENCE):+.1f} pcm from the reference")

## Mesh refinement, all three kernels

In [ ]:
rows = []
for sub in (1, 2, 4, 8):
    g = build(sub)
    ks = {k: openndm.Model(g, library, settings).solve(kernel=k).k_eff
          for k in ("fdm", "nem", "sanm")}
    rows.append((sub, g.n_nodes, ks))

header = f"{'nodes/asm':>9} {'nodes':>6}" + "".join(f"{k.upper():>24}" for k in ("fdm", "nem", "sanm"))
print(header)
for sub, n, ks in rows:
    cells = "".join(f"{v:14.6f} ({1e5 * (v - REFERENCE):+7.1f})" for v in ks.values())
    print(f"{sub:>9} {n:>6}{cells}")

Read the top-right cell first. **SANM on a 20 cm mesh is 3.7 pcm from the
reference**, which is what a nodal method is for.

Two other things in that table are worth understanding.

SANM and NEM both converge to 1.029527 and stay there from four nodes per
assembly onward, agreeing with each other to 0.08 pcm. They share only the
transverse leakage fit — NEM closes its two-node problem with quartic
polynomials and the coarse-mesh outer-face currents, SANM with analytic basis
functions and the node-average constraint — so they have no reason to agree
that closely unless both are right.

FDM is non-monotone, and that is not a defect. It starts 373 pcm high, crosses
the converged value between 10 and 5 cm, and approaches from below at second
order. The crossover is two error terms of opposite sign: on the coarsest mesh
the 20 cm reflector is a single, badly under-resolved node.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
subs = [r[0] for r in rows]
for kernel in ("fdm", "nem", "sanm"):
    errors = [1e5 * (r[2][kernel] - REFERENCE) for r in rows]
    ax.plot(subs, errors, "o-", label=kernel.upper())
ax.axhline(0, color="k", lw=0.8)
ax.axhspan(-100, 100, color="grey", alpha=0.15,
           label="100 pcm acceptance")
ax.set_xscale("log", base=2)
ax.set_xticks(subs); ax.set_xticklabels(subs)
ax.set_xlabel("nodes per assembly")
ax.set_ylabel("error vs reference [pcm]")
ax.set_title("IAEA-2D: nodal kernels converge from the coarsest mesh")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()

## The power distribution

In [ ]:
radial = result.radial_power()
print("assembly power, mean 1.0 over powered positions\n")
for row in radial[::-1]:
    print("  " + " ".join("  .  " if v == 0 else f"{v:5.3f}" for v in row))
print(f"\nF_dH = {result.f_dh:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.4))
masked = np.ma.masked_where(radial <= 0, radial)
im = ax.imshow(masked, origin="lower", cmap="viridis")
fig.colorbar(im, ax=ax, label="relative assembly power")
for (j, i), v in np.ndenumerate(radial):
    if v > 0:
        ax.text(i, j, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="w" if v < radial.max() * 0.6 else "k")
ax.set_title(f"IAEA-2D radial power (SANM), $F_{{\\Delta H}}$ = {result.f_dh:.3f}")
ax.set_xlabel("assembly, x"); ax.set_ylabel("assembly, y")
fig.tight_layout()

The four depressions are the rodded positions, at the core centre and at the
symmetric locations that make nine rodded assemblies in the full core.

## Checks worth running on any core

In [ ]:
model = openndm.Model(build(1), library, settings)
model.solve()

# The map is symmetric about the diagonal, so the power must be too. This is
# the cheapest possible check on the x and y indexing paths.
power = model.solve().radial_power()
print(f"power symmetric about the diagonal to "
      f"{np.abs(power - power.T).max():.2e}")

# Every node conserves neutrons.
print(f"worst neutron balance residual: "
      f"{np.abs(model.neutron_balance()).max():.2e}")

# The adjoint shares the spectrum of the forward operator.
forward = model.solve(kernel="fdm").k_eff
adjoint = model.solve_adjoint(kernel="fdm").k_eff
print(f"adjoint k_eff differs from forward by "
      f"{1e5 * abs(forward - adjoint):.3f} pcm")

## Next

`benchmarks/iaea3d/` extrudes this map to 380 cm with a partially inserted
control bank, and reproduces its own published reference to 45 pcm. Its
`README` explains the two transcription errors that had to be found first,
including why the specification's "80 cm" is the height of the rod tips above
the bottom of the core rather than an insertion depth from the top — a reading
worth 1700 pcm.